## Prescription Parser Agent

In [1]:
import uuid
import base64
from flask import Flask, request, jsonify
from dotenv import load_dotenv, find_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver  
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List
from datetime import datetime as dt

load_dotenv(find_dotenv(), override=True)

True

In [2]:
class PrescInfo(BaseModel):
    date: str = Field(description="Date formatted in MM-DD-YYYY")
    prescription_name: str = Field(description="Name of prescription given exactly.")
    active_ingrediants: List[str] = Field(description="list of active ingrediants in drug")
    patient_first_name: str = Field(description="First name of patient")
    patient_last_name: str = Field(description="Last name of patient")
    patient_DOB: str = Field(description="Date of birth of patient in MM/DD/YYYY")
    medication_details: str = Field(description="Details and instructions on usage")
    num_refills: float = Field(description="Maximum number of refills allowed")
    is_signed: bool = Field(description="True if the description is valid and signed, false otherwise")
    fulltext: str = Field(description="All text on prescription for analysis")
    

In [3]:
with open('./data/proair_presc.png', 'rb') as imgfp:
    img_str = base64.b64encode(imgfp.read()).decode("utf-8")

In [4]:
presc_processor_prompt = SystemMessage(content=("You are an expert prescription parser.\n" 
    "Your job is to parse the prescription into its components for further analysis."))

presc_human_data = HumanMessage(content=[
    {"type": "text", "text": "Parse the following prescription into the output format"},
    {
        "type": "image_url",
        "image_url": {"url":f"data:image/jpeg;base64,{img_str}"}
    }
])

messages = [
    presc_processor_prompt, presc_human_data
]


In [5]:
llm = ChatOpenAI(temperature=0.2, model='gpt-5.4')
threadID = f"presc-{uuid.uuid4().hex[:8]}"
mem = InMemorySaver()
mem.delete_thread(threadID)
presc_agent = create_agent(
    model=llm, 
    response_format=PrescInfo,
    checkpointer=mem
)

In [6]:
res = presc_agent.invoke({"messages": messages}, config={"configurable": {"thread_id": threadID}})

In [7]:
receipt_info: PrescInfo = res["structured_response"]
print(receipt_info)

date='05-24-2024' prescription_name='ProAir HFA (albuterol sulfate)' active_ingrediants=['albuterol sulfate'] patient_first_name='Emily' patient_last_name='Harris' patient_DOB='07/12/1998' medication_details='90 mcg inhalation aerosol. Inhale 2 puffs by mouth every 4 to 6 hours as needed for wheezing or shortness of breath. Dispense: 1 inhaler.' num_refills=2.0 is_signed=True fulltext='Michael J. Thompson, MD\nINTERNAL MEDICINE\n1234 Oakview Drive, Suite 200\nRaleigh, NC 27609\n(919) 555-2678\nNPI: 1234567890\nDate: 5/24/24\nPatient Name: Emily J. Harris\nDOB: 7/12/1998\nRx\nProAir HFA (albuterol sulfate)\n90 mcg inhalation aerosol\nInhale 2 puffs by mouth every 4 to 6 hours as needed for wheezing or shortness of breath.\nDispense: 1 inhaler\nRefills: 2\nSignature'


### Patient Look-Up

In [8]:
from datetime import datetime

from psql import connect_str
import psycopg as pg

with pg.connect(connect_str) as conn:
    with conn.cursor() as cur:
        fname = receipt_info.patient_first_name
        lname = receipt_info.patient_last_name
        dob = datetime.strptime(receipt_info.patient_DOB, '%m/%d/%Y').date()
        query = f"SELECT * FROM patients WHERE first_name = '{fname}' AND last_name = '{lname}' AND date_of_birth = %s"
        cur.execute(query, (dob,))
        existingRecords = cur.fetchall()
        for row in existingRecords:
            print(row)
        
        if len(existingRecords) == 0:
            newUUID = uuid.uuid4()
            updateQuery = f"INSERT INTO patients (first_name, last_name, date_of_birth) VALUES ('{fname}', '{lname}', %s)"
            cur.execute(updateQuery, (dob,))
            
        

('PostgreSQL 16.14 (Ubuntu 16.14-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit',)
(UUID('d951da4a-f5f2-4cbf-8b92-e327d8efa921'), 'Emily', 'Harris', datetime.date(1998, 7, 12), datetime.datetime(2026, 7, 1, 15, 8, 36, 139522, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')), datetime.datetime(2026, 7, 1, 15, 8, 36, 139522, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')))


## Pre QA

In [ ]:

dataloader._initialize_and_load_data()

Loaded proair hfa reference data into memory.


In [ ]:
drug_name = "ProAir HFA"
drug_results = dataloader.drug_collection.query(
        query_texts=[f"warnings and interactions, not recommended, other drugs, contraindications for {drug_name}"],
        n_results=10,
        where={"drug_name": drug_name.lower()}
    )

AttributeError: module 'dataloader' has no attribute 'drug_collection'

In [ ]:
print("\n--- Retrieved Drug Context ---")
print(drug_results["documents"])
print(len(drug_results["documents"]))


--- Retrieved Drug Context ---
[['Before using ProAir HFA\nIn deciding to use a medicine, the risks of taking the medicine must be weighed against the good it will do. This is a decision you and your doctor will make. For this medicine, the following should be considered:', 'Uses\nBefore taking\nDosage\nWarnings\nSide effects\nBrand names\nFAQ\nUses for ProAir HFA\nAlbuterol is used to treat or prevent bronchospasm in patients with asthma, bronchitis, emphysema, and other lung diseases. It is also used to prevent bronchospasm caused by exercise.', 'Side Effects of ProAir HFA\nAlong with its needed effects, a medicine may cause some unwanted effects. Although not all of these side effects may occur, if they do occur they may need medical attention.', 'Does ProAir HFA interact with my other drugs?\nEnter medications to view a detailed interaction report using our Drug Interaction Checker.', "Detailed ProAir HFA dosage information\nPrecautions while using ProAir HFA\nIt is very important